# External validation using the external validators

This notebook evaluates the external validity of latent symptom dimensions derived from autoencoder, EFA, PCA, and kernel PCA models. The analysis uses the ten manuscript validators to examine how well the latent dimensions relate to clinical, cognitive, family, school, developmental, peer, and medication-related outcomes.

The workflow includes:

1. loading the ABCD baseline validator variables and converting them into the ten validation outcomes used in the manuscript;
2. loading sex, site, and age covariates and merging them with the validator table;
3. loading CBCL questionnaire data and latent factor scores saved in `../output/`;
4. combining each method's factor scores with the validator and covariate data by participant ID;
5. comparing autoencoder and EFA validation performance across one to five dimensions;
6. saving validation summaries and figures under `../output/validator_comparison_between_efa_ae/`.


In [ ]:
# Imports and global settings
from pathlib import Path
import numpy as np
import pandas as pd

from validators import (
    build_10_items_validators,
    build_validators_baseline,
    build_model_scores,
    get_scores_by_k,
    compare_efa_poly_vs_ae_poly,
    build_10_items_validators,
)
from factor_analyzer import FactorAnalyzer
from sklearn.preprocessing import StandardScaler
from IPython.display import Image, display
%load_ext autoreload
%autoreload 2

seed = 52
encoding_dim = 5

code_dir = Path.cwd()
data_path = code_dir.parent / "data"
output_dir = code_dir.parent / "output"
validators_out_dir = output_dir / "validators"
validators_out_dir.mkdir(parents=True, exist_ok=True)

# ABCD source paths. Keep these unchanged if your local data paths are the same.
ROOT = Path(r"G:\ABCD\abcd-data-release-5.1")
DICT = Path(r"G:\ABCD\datadict51.xlsx")

print("code_dir:", code_dir)
print("output_dir:", output_dir)

In [ ]:
# The ten external validators used in the manuscript
VALIDATORS = {
    "dev_delay": ["devhx_20_p", "devhx_21_p"],
    "fes_conflict": [f"fes_youth_q{i}" for i in range(1, 10)],
    "n_friends": ["resiliency5a_y", "resiliency6a_y", "resiliency5b_y", "resiliency6b_y"],
    "school_conn": [f"school_{i}_y" for i in (2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 17)],
    "avg_grades": ["kbi_p_grades_in_school"],
    "fluid_cog": ["nihtbx_fluidcomp_uncorrected"],
    "cryst_cog": ["nihtbx_cryst_uncorrected"],
    "mh_service": ["kbi_p_c_mh_sa"],
    "med_history": ["medhx_1a", "medhx_1b"],
    "brought_meds": ["brought_medications"],
}

list(VALIDATORS.keys())

In [ ]:
# Build or load the baseline validator table
validators_baseline_csv = validators_out_dir / "validators_baseline.csv"

if validators_baseline_csv.exists():
    validators_raw = pd.read_csv(validators_baseline_csv, encoding="utf-8")
    print("Loaded existing:", validators_baseline_csv)
else:
    _, validators_raw = build_validators_baseline(
        root=ROOT,
        dict_path=DICT,
        validators=VALIDATORS,
        eventname="baseline_year_1_arm_1",
        out_dir=validators_out_dir,
        dict_sheet=None,
        dict_engine="openpyxl",
        verbose=True,
        wide_table_name="validators_baseline.csv",
        output_separate_dirs=True,
    )
    print("Saved:", validators_baseline_csv)

# Convert the raw ABCD items into the ten manuscript validators.
validators_df = build_10_items_validators(validators_csv=str(validators_baseline_csv))
print(validators_df.shape)
validators_df.head()

In [ ]:
# Load covariates used for deconfounding
COVARIATES = {
    "sex": ["demo_gender_id_v2"],
    "site": ["site_id_l"],
    "age": ["interview_age"],
}

covariates_csv = validators_out_dir / "validator_covariates_baseline.csv"

if covariates_csv.exists():
    covariates_df = pd.read_csv(covariates_csv, encoding="utf-8")
    print("Loaded existing:", covariates_csv)
else:
    _, covariates_df = build_validators_baseline(
        root=ROOT,
        dict_path=DICT,
        validators=COVARIATES,
        eventname="baseline_year_1_arm_1",
        out_dir=validators_out_dir,
        dict_sheet=None,
        dict_engine="openpyxl",
        verbose=True,
        wide_table_name="validator_covariates_baseline.csv",
        output_separate_dirs=True,
    )
    print("Saved:", covariates_csv)

# Merge covariates into the ten-validator table.
# The sex column may be named demo_gender_id_v2 or demo_gender_id_v2_x depending on the data dictionary merge.
sex_col = "demo_gender_id_v2_x" if "demo_gender_id_v2_x" in covariates_df.columns else "demo_gender_id_v2"
required_cov_cols = ["src_subject_id", sex_col, "site_id_l", "interview_age"]

validators_df = validators_df.merge(
    covariates_df[required_cov_cols],
    on="src_subject_id",
    how="left",
)

validators_df = validators_df.rename(columns={sex_col: "sex"})
print(validators_df.shape)
validators_df.head()

In [ ]:
# Load CBCL questionnaire data and factor scores from ../output/
data_file = data_path / "cbcl_data_remove_low_frequency.csv"
if not data_file.exists():
    raise FileNotFoundError(f"Could not find {data_file}")

qns = pd.read_csv(data_file, encoding="utf-8")

factor_files = {
    "AE": output_dir / f"latent_factors_ae_dim{encoding_dim}.csv",
    "EFA": output_dir / f"latent_factors_efa_dim{encoding_dim}.csv",
    "PCA": output_dir / f"latent_factors_pca_dim{encoding_dim}.csv",
    "KPCA": output_dir / f"latent_factors_kpca_dim{encoding_dim}.csv",
}

missing = [str(path) for path in factor_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing factor score files in ../output:/" + "\n" + "\n".join(missing))

results_by_method = {}
for method, path in factor_files.items():
    scores = pd.read_csv(path, encoding="utf-8")
    scores = build_model_scores(qns, scores)
    results_by_method[method] = scores.merge(validators_df, on="src_subject_id", how="inner")
    print(method, results_by_method[method].shape)

In [ ]:
def make_efa(k):
    return FactorAnalyzer(n_factors=k, rotation="geomin_obl")

X = qns.iloc[:, 1:].values

# Standardize the data
scaler = StandardScaler()
# scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

efa_scores_by_k = get_scores_by_k(make_efa, X_scaled, subject_id = qns.iloc[:, 0], Kmax=5, prefix="efa")
AE_scores = build_model_scores(qns, results_by_method["AE"].iloc[:, 1:6])

out = compare_efa_poly_vs_ae_poly(
    factors_scores=AE_scores,
    efa_scores_by_k=efa_scores_by_k,
    validators_csv="../output/validators/validators_baseline.csv",
    save_dir="../output/validator_comparison_between_efa_ae",
    degree=1,                         # 不造多项式
    use_poly_features=False,          # 关键
    model_reg="gbrt",                 # 连续：GBDT
    model_clf="gbrt",                 # 二分类：GBDT
    random_seed=6,
)

display(Image(filename=out["paths"]["grid_png"]))   # R² vs K 曲线图
print("Summary CSV:", out["paths"]["summary_csv"])  # 每个outcome×K的AE/EFA R²与差异、提升率、p值